# OCR Pipeline Testing Notebook

Interactive testing of each pipeline component.

In [4]:
import sys

sys.path.insert(0, "..")

from ocr_manga_title.config import load_config, load_ocr_config

config = load_config("../configs.toml")
ocr_config = load_ocr_config("../ocrs.yaml")

print("App Config:")
print(f"  images_path: {config.images_path}")
print(f"  openrouter model: {config.openrouter.default_model}")
print("\nOCR Models:")
for name, cfg in ocr_config.items():
    print(f"  {name}: enabled={cfg.enabled}, language={cfg.language}")

App Config:
  images_path: /Users/rconsuegra/Pictures
  openrouter model: google/gemini-2.5-flash

OCR Models:
  ocr_manga_title: enabled=True, language=ja
  tesseract: enabled=True, language=en
  paddle: enabled=False, language=en
  easyocr: enabled=False, language=en
  glm_ocr: enabled=False, language=en


## 1. Run manga-ocr

In [5]:
from ocr_manga_title.models.ocr_manga_title_model import MangaOCRModel

model = MangaOCRModel(ocr_config["ocr_manga_title"])
print(f"Available: {model.is_available}")

if model.is_available:
    import glob
    images = sorted(glob.glob(str(config.images_path / "*.png")))[:3]
    for img_path in images:
        print(f"\n--- {img_path} ---")
        result = model.run(img_path)
        print(f"Raw text: {result.raw_text[:200]}")
        print(f"Confidence: {result.confidence}")
        print(f"Time: {result.processing_time_ms}ms")

Available: True


## 2. Run Tesseract

In [6]:
from ocr_manga_title.models.tesseract_model import TesseractModel

model = TesseractModel(ocr_config["tesseract"])
print(f"Available: {model.is_available}")

if model.is_available:
    for img_path in images:
        print(f"\n--- {img_path} ---")
        result = model.run(img_path)
        print(f"Raw text: {result.raw_text[:200]}")
        print(f"Confidence: {result.confidence}")
        print(f"Time: {result.processing_time_ms}ms")

Available: False


## 3. LLM Extraction

In [ ]:
from ocr_manga_title.postprocess.llm_extractor import LLMExtractor

extractor = LLMExtractor(config.openrouter, prompt_path="../prompts/llm/extract_title_v1.md")

sample_text = "\u30ef\u30f3\u30d4\u30fc\u30b9 One Piece ISBN 978-4-08-872509-4"
result = extractor.extract(sample_text)
print(f"title_en: {result.title_en}")
print(f"title_ja: {result.title_ja}")
print(f"code: {result.code}")
print(f"confidence: {result.confidence}")

## 4. Rule Matching

In [ ]:
from ocr_manga_title.postprocess.rule_matcher import RuleMatcher

matcher = RuleMatcher()
text = "Published as ISBN 978-4-06-319310-8"
result = matcher.match(text)
print(f"code: {result.code}")
print(f"confidence: {result.confidence}")

## 5. Full Pipeline

In [ ]:
from ocr_manga_title.engine import OCREngine

engine = OCREngine(config, ocr_config)
if images:
    result = engine.process(images[0])
    print(f"Input: {result.input_path}")
    print(f"Extracted: {result.extracted}")
    print(f"OCR Results: {len(result.ocr_results)}")
    for r in result.ocr_results:
        print(f"  {r.model_name}: conf={r.confidence}, time={r.processing_time_ms}ms")
        if r.error:
            print(f"    ERROR: {r.error}")
    print(f"Errors: {result.errors}")

## 6. Batch Test

In [ ]:
from pathlib import Path

import pandas as pd

results = []
for img in images[:10]:
    r = engine.process(img)
    results.append({
        "image": Path(r.input_path).name,
        "title_en": r.extracted.title_en if r.extracted else None,
        "title_ja": r.extracted.title_ja if r.extracted else None,
        "code": r.extracted.code if r.extracted else None,
        "confidence": r.extracted.confidence if r.extracted else 0,
        "errors": len(r.errors),
    })

df = pd.DataFrame(results)
df